# Introduccion al Parser de Pseudocodigo

**Modulo:** Parser  
**Objetivo:** Familiarizarse con el parser y su funcionamiento  
**Duracion estimada:** 25 minutos

---

## Contenido

1. [Setup](#setup)
2. [Arquitectura del Parser](#arquitectura-del-parser)
3. [Componentes del Sistema](#componentes-del-sistema)
4. [Ejemplo Basico](#ejemplo-basico)
5. [Explorando el AST](#explorando-el-ast)
6. [Ejercicios](#ejercicios)

---

## 1. Setup

In [ ]:
# Agregar path del proyecto
import sys
sys.path.insert(0, '../..')

# Imports necesarios
from app.core.parser import (
    PseudocodeParser,
    parse_pseudocode,
    ASTBuilder,
    SemanticAnalyzer,
    ASTValidator
)
from app.core.parser.ast_nodes import (
    ProgramNode,
    AlgorithmNode,
    ForLoopNode,
    WhileLoopNode,
    IfStatementNode
)

# Para visualizacion
from pprint import pprint

print("Setup completado")

---

## 2. Arquitectura del Parser

El parser de pseudocodigo es el componente central del modulo de parsing. Su funcion principal es:

1. **Leer codigo pseudocodigo** en sintaxis estilo PSeInt
2. **Generar un Parse Tree** utilizando la gramatica definida en Lark
3. **Construir el AST** mediante el `ASTBuilder`

### Flujo de Procesamiento

```
Codigo Pseudocodigo
        |
        v
+-------------------+
|  PseudocodeParser |
+-------------------+
        |
        v
+-------------------+
|   Lark Grammar    |
|  (pseudocode.lark)|
+-------------------+
        |
        v
+-------------------+
|   Parse Tree      |
+-------------------+
        |
        v
+-------------------+
|   ASTBuilder      |
+-------------------+
        |
        v
+-------------------+
|   AST (Nodos)     |
+-------------------+
```

---

## 3. Componentes del Sistema

| Componente | Ubicacion | Funcion |
|------------|-----------|----------|
| `PseudocodeParser` | `parser/pseudocode_parser.py` | Parser principal con Lark |
| `ASTBuilder` | `parser/ast_builder.py` | Transforma parse tree a AST |
| `ast_nodes` | `parser/ast_nodes.py` | Definicion de nodos del AST |
| `SemanticAnalyzer` | `parser/semantic_analyzer.py` | Validacion semantica |
| `ASTValidator` | `parser/validator.py` | Validacion estructural |

### Tipos de Nodos AST

| Nodo | Descripcion |
|------|-------------|
| `ProgramNode` | Raiz del programa |
| `AlgorithmNode` | Definicion del algoritmo |
| `BlockNode` | Bloque begin...end |
| `ForLoopNode` | Ciclo for |
| `WhileLoopNode` | Ciclo while |
| `IfStatementNode` | Condicional if...then...else |
| `AssignmentNode` | Asignacion variable <- expresion |
| `CallStatementNode` | Llamada a funcion/procedimiento |

---

## 4. Ejemplo Basico

### Parseando un Algoritmo Simple

In [ ]:
# Codigo de ejemplo: Algoritmo de suma
codigo_suma = """
algorithm suma(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
"""

# Crear instancia del parser
parser = PseudocodeParser()

# Parsear el codigo
ast = parser.parse(codigo_suma)

# Mostrar informacion basica
print("RESULTADO DEL PARSING")
print(f"Algoritmo: {ast.algorithm.name}")
print(f"Parametros: {[p.name for p in ast.algorithm.parameters]}")
print(f"Numero de statements: {len(ast.algorithm.body.statements)}")

**Salida esperada:**
```
RESULTADO DEL PARSING
Algoritmo: suma
Parametros: ['n']
Numero de statements: 3
```

### Explorando la Estructura del AST

In [ ]:
# Explorar cada statement del algoritmo
print("STATEMENTS EN EL ALGORITMO:")
for i, stmt in enumerate(ast.algorithm.body.statements, 1):
    print(f"{i}. Tipo: {stmt.node_type.value}")
    print(f"   Clase: {type(stmt).__name__}")
    print()

---

## 5. Explorando el AST

### 5.1 Sintaxis Soportada

El parser soporta la siguiente sintaxis basada en PSeInt:

| Categoria | Ingles | Espanol |
|-----------|--------|----------|
| Algoritmo | `algorithm` | `algoritmo`, `proceso`, `funcion` |
| Inicio | `begin` | `inicio` |
| Fin | `end` | `fin`, `finproceso`, `finalgoritmo` |
| Para | `for` | `para` |
| Mientras | `while` | `mientras` |
| Si | `if` | `si` |
| Llamar | `call` | `llamar` |
| Retornar | `return` | `retornar` |

### 5.2 Operadores

| Tipo | Operadores |
|------|------------|
| Asignacion | `<-` |
| Aritmeticos | `+`, `-`, `*`, `/`, `^`, `%` |
| Comparacion | `<`, `>`, `<=`, `>=`, `=`, `!=` |
| Logicos | `and`, `or`, `not` |

### Ejemplo: Algoritmo con Ciclos Anidados

In [ ]:
# Algoritmo mas complejo: Bubble Sort
codigo_bubble = """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
"""

ast_bubble = parser.parse(codigo_bubble)

print(f"Algoritmo: {ast_bubble.algorithm.name}")
print(f"Parametros: {[(p.name, p.param_type) for p in ast_bubble.algorithm.parameters]}")

# Encontrar el ciclo for externo
for stmt in ast_bubble.algorithm.body.statements:
    if isinstance(stmt, ForLoopNode):
        print(f"\nCiclo FOR encontrado:")
        print(f"  Variable: {stmt.variable}")
        print(f"  Cuerpo tiene {len(stmt.body.statements)} statement(s)")

### Serializacion del AST a Diccionario

In [ ]:
# Convertir AST a diccionario
ast_dict = ast.to_dict()

print("AST COMO DICCIONARIO:")
print(f"Tipo: {ast_dict['type']}")
print(f"Algoritmo: {ast_dict['algorithm']['name']}")
print(f"Parametros: {ast_dict['algorithm']['parameters']}")

### Validacion del AST

In [ ]:
# Validar estructura del AST
validator = ASTValidator(max_depth=20, max_nodes=10000)
result = validator.validate(ast)

print("VALIDACION DEL AST:")
print(f"Es valido: {result.is_valid}")
print(f"Errores: {result.errors}")
print(f"Warnings: {result.warnings}")

In [ ]:
# Usar la funcion helper parse_pseudocode
codigo_recursivo = """
algorithm factorial(n)
begin
    if (n <= 1) then
        return 1
    end
    return n * factorial(n - 1)
end
"""

ast_factorial = parse_pseudocode(codigo_recursivo)

print(f"Algoritmo: {ast_factorial.algorithm.name}")
print(f"Es recursivo: contiene llamada a si mismo")

---

## 6. Ejercicios

### Ejercicio 1: Parsear un Algoritmo de Busqueda

Parsea el siguiente algoritmo y explora su AST:

In [ ]:
# Ejercicio 1: Parsea este algoritmo
codigo_busqueda = """
algorithm linearSearch(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
"""

# Tu codigo aqui
ast_busqueda = parse_pseudocode(codigo_busqueda)

# Imprime el nombre y los parametros
print(f"Algoritmo: {ast_busqueda.algorithm.name}")
print(f"Numero de parametros: {len(ast_busqueda.algorithm.parameters)}")
for p in ast_busqueda.algorithm.parameters:
    print(f"  - {p.name} (tipo: {p.param_type})")

### Ejercicio 2: Contar Tipos de Nodos

Cuenta cuantos nodos de cada tipo hay en el AST:

In [ ]:
# Ejercicio 2: Contar tipos de nodos
from collections import Counter

def count_node_types(node, counter=None):
    if counter is None:
        counter = Counter()
    
    counter[type(node).__name__] += 1
    
    # Recorrer hijos
    if hasattr(node, 'body') and node.body:
        if hasattr(node.body, 'statements'):
            for stmt in node.body.statements:
                count_node_types(stmt, counter)
        else:
            count_node_types(node.body, counter)
    
    return counter

# Contar nodos en bubbleSort
counts = count_node_types(ast_bubble.algorithm)
print("TIPOS DE NODOS EN BUBBLE SORT:")
for node_type, count in counts.most_common():
    print(f"  {node_type}: {count}")

---

## 7. Tips y Resumen

### Tips

- **Usa `parse_pseudocode()`** para parsing rapido sin crear instancia
- **Explora con `to_dict()`** para ver estructura completa del AST
- **Valida con `ASTValidator`** antes de analizar algoritmos grandes
- **Los parametros de array** se detectan automaticamente con `[]`

### Resumen

Has aprendido:
- Como funciona la arquitectura del parser
- Parsear codigo pseudocodigo a AST
- Explorar la estructura de nodos
- Validar y serializar el AST
- Trabajar con diferentes tipos de algoritmos

---

## Proximos Pasos

- **ast_visualization.ipynb**: Visualizar la estructura del AST
- **grammar_experiments.ipynb**: Experimentar con la gramatica Lark

**Siguiente modulo recomendado**: `02_complexity_analysis/` para analizar complejidad